# Staggered Evacuation Sensitivity Analysis

This notebook performs sensitivity analysis on staggered EV evacuation using three Beta distribution scenarios:
1. **early_evac** (α=2, β=5): Early evacuation spike
2. **delayed_evac** (α=5, β=2): Delayed evacuation
3. **uniform_evac** (α=1, β=1): Uniform evacuation

The staggered start parameter R_{ilt} represents the number of EVs entering the network at location i with charge level l at time t.


In [ ]:
# A tiny test before running the entire code
import pyomo.environ as pe
import pyomo.opt as po

solver = po.SolverFactory("gurobi")
print("Gurobi available:", solver.available())


In [ ]:
import pyomo.environ as pe
import pyomo.opt as po
from pyomo.opt import SolverFactory
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from collections import defaultdict
import pandas as pd
import time
import math
import os
import sys

# Add code directory to path to import distributions module
sys.path.insert(0, '../code')
from distributions import StaggeredEvacuationDistributions


In [ ]:
solver = po.SolverFactory('gurobi')
# Set the environment variable for the license file if needed
# os.environ['GRB_LICENSE_FILE'] = '/path/to/your/gurobi.lic'


In [ ]:
# Path definition
Inputpath = "../input/"
Outputpath = "../output/numerical_results/"


In [ ]:
# Basic fixed parameters
L = 27  # Number of charge levels (196/7.5)
T = 2   # Evacuation time window
G = 3.7 # Cost of charging capacity installed at each node (million $/thousand cars)
B = 10  # Budget (million $)

# Load network data
df_arc = pd.read_excel(os.path.join(Inputpath, "arcs_sonoma_15min.xlsx"), header=0)
df_node = pd.read_excel(os.path.join(Inputpath, "nodes_sonoma_15min.xlsx"), header=0)
df_char_damFac = pd.read_csv(os.path.join(Inputpath, "charging_cap_damFac_3hrs.csv"), header=0)
df_road_damFac = pd.read_csv(os.path.join(Inputpath, "road_cap_damFac_3hrs.csv"), header=0)
df_S_nodes = pd.read_excel(os.path.join(Inputpath, "safe_nodes_15min.xlsx"), header=0)
df_S_nodes = df_S_nodes[df_S_nodes['Safe'] == 1]


In [ ]:
# Get scenario probabilities
cases_num = max(df_char_damFac['Scenario'])
prob_dict = dict()
for k in range(cases_num):
    prob_dict[k] = df_char_damFac.groupby(['Scenario']).first()['Probability'][k+1]
print(f"Number of scenarios: {cases_num}")
print(f"Scenario probabilities: {prob_dict}")


## Define Network Sets and Parameters


In [ ]:
# Define network sets
# The set of nodes in the safety area
N_s = set()
for i in range(len(df_S_nodes)):
    N_s.add(df_S_nodes.iloc[i]['Nodes'])

# Set of all nodes
N = set()
for i in range(len(df_node)):
    N.add(df_node['Nodes'][i])

# Set of arcs and road capacity
A = set()
road_cap = dict()

for i in range(len(df_arc)):
    node0 = df_arc['From'][i]
    node1 = df_arc['To'][i]
    A.add((node0, node1))
    road_cap[((node0, node1))] = df_arc['Capacity'][i]

print(f"Number of nodes: {len(N)}")
print(f"Number of safe nodes: {len(N_s)}")
print(f"Number of arcs: {len(A)}")


In [ ]:
# Define incoming and outgoing arc sets for each node
Vi = defaultdict(set)
Vo = defaultdict(set)
for (i, j) in A:
    Vi[j].add(i)  # incoming set
    Vo[i].add(j)  # outgoing set


In [ ]:
# Stochastic parameters: road capacity
def c_random(A, df_road_damFac, cases_num, road_cap, T):
    c_tuple = dict()
    gk = df_road_damFac.groupby(['Scenario'])

    for k in range(cases_num):
        sce_sub = gk.get_group(k+1).copy()
        for r in range(sce_sub.shape[0]):
            curr_edge = sce_sub.iloc[r]
            for t in range(4*T+1):
                if math.isnan(curr_edge['Time']):
                    c_tuple[((int(curr_edge['From']), int(curr_edge['To'])), k), t] = road_cap[(int(curr_edge['From']), int(curr_edge['To']))]
                elif t/4 < curr_edge['Time']:
                    c_tuple[((int(curr_edge['From']), int(curr_edge['To'])), k), t] = road_cap[(int(curr_edge['From']), int(curr_edge['To']))]
                else:
                    c_tuple[((int(curr_edge['From']), int(curr_edge['To'])), k), t] = 0
    return c_tuple


In [ ]:
# Stochastic parameters: charging capacity factors
def s_random(df_char_damFac, cases_num, T):
    xi_s = dict()
    gk = df_char_damFac.groupby(['Scenario'])

    for k in range(cases_num):
        sce_sub = gk.get_group(k+1).copy()
        for r in range(sce_sub.shape[0]):
            curr_node = sce_sub.iloc[r]
            for t in range(4*T+1):
                if math.isnan(curr_node['Time']):
                    xi_s[(int(curr_node['Nodes']), k), t] = 1
                elif t/4 < curr_node['Time']:
                    xi_s[(int(curr_node['Nodes']), k), t] = 1
                else:
                    xi_s[(int(curr_node['Nodes']), k), t] = 0
    return xi_s


In [ ]:
road_ca = c_random(A, df_road_damFac, cases_num, road_cap, T)  # road capacity random parameter
xi_s = s_random(df_char_damFac, cases_num, T)  # charging capacity random parameter


## Define Staggered Evacuation Scenarios


In [ ]:
# Define staggered evacuation scenarios with flexible parameters
scenarios = {
    "early_evac": {"alpha": 2, "beta": 5},    # Right-skewed: early evacuation spike
    "delayed_evac": {"alpha": 5, "beta": 2},  # Left-skewed: delayed evacuation
    "uniform_evac": {"alpha": 1, "beta": 1}   # Uniform staggered evacuation
}

print("Staggered Evacuation Scenarios:")
for scenario_name, params in scenarios.items():
    print(f"  {scenario_name}: α={params['alpha']}, β={params['beta']}")


## Generate R_ilt Distributions and Statistics


In [ ]:
# Load car initialization data (example with even battery distribution)
# You can modify this to load different car distributions
df_car_init = pd.read_csv(
    os.path.join(
        Inputpath,
        "car_charge_distribution",
        "CarNumberInitial_halfEV_L26even_battery.csv"
    ),
    header=0
)

# Convert dataframe to dictionary
car_num = dict()
for line in range(len(df_car_init)):
    car_num[(df_car_init['Levels'][line], df_car_init['Nodes'][line])] = df_car_init['CarNumbers'][line]

# Create E_il (total EVs by location and charge level)
node_list = sorted(df_car_init["Nodes"].unique())
assert set(node_list) == set(N), (
    f"Node mismatch: car distribution has {sorted(set(node_list) - set(N))}, "
    f"model has {sorted(set(N) - set(node_list))}"
)

E_il_array = np.zeros((len(node_list), L))

for node_idx, node in enumerate(node_list):
    for level in range(L):
        E_il_array[node_idx, level] = car_num.get((level, node), 0)


In [ ]:
# Generate R_ilt distributions for all scenarios
# R_ilt shape will be (num_nodes, num_levels, T_periods) for each scenario

# Generate staggered start distributions for all scenarios
R_distributions = StaggeredEvacuationDistributions.generate_beta_scenarios(
    E_il=E_il_array,
    T=4*T,  # Use same time discretization as existing notebooks
    scenarios=scenarios,
    random_state=42
)
for scenario_name, R_ilt in R_distributions.items():
    assert np.allclose(R_ilt.sum(axis=2), E_il_array), (
        f"R_ilt does not sum to E_il for {scenario_name}"
    )

print("Generated R_ilt distributions for all scenarios")
for scenario_name, R_ilt in R_distributions.items():
    print(f"  {scenario_name}: shape={R_ilt.shape}, dtype={R_ilt.dtype}")


In [ ]:
# Calculate and display statistics for each scenario
print("\n" + "="*80)
print("Staggered Evacuation Distribution Statistics")
print("="*80)

stats_list = []
for scenario_name, R_ilt in R_distributions.items():
    stats = StaggeredEvacuationDistributions.distribution_statistics(R_ilt, scenario_name)
    stats_list.append(stats)
    
    print(f"\n{scenario_name.upper()}:")
    print(f"  Total EVs: {stats['total_evs']}")
    print(f"  Mean evacuation time: {stats['mean_time']:.2f}")
    print(f"  Peak time period: {stats['peak_time']}")
    print(f"  Peak evacuation rate: {stats['peak_value']} EVs/period")
    print(f"  Time to 80% evacuation: {stats['time_to_80_percent']} periods")
    print(f"  Coefficient of variation: {stats['coefficient_of_variation']:.4f}")

# Create statistics dataframe
df_stats = pd.DataFrame(stats_list)
print("\n" + "="*80)
print(df_stats.to_string(index=False))


## Visualize R_ilt Distributions


In [ ]:
# Visualize evacuation profiles across time for all three scenarios
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Staggered Evacuation Distributions: R_{ilt} Over Time', fontsize=14, fontweight='bold')

for idx, (scenario_name, R_ilt) in enumerate(R_distributions.items()):
    # Sum over all nodes and charge levels to get evacuation profile over time
    evacuation_profile = np.sum(R_ilt, axis=(0, 1))
    
    axes[idx].bar(range(len(evacuation_profile)), evacuation_profile, color=['steelblue', 'coral', 'lightgreen'][idx], alpha=0.7)
    axes[idx].set_title(f'{scenario_name}\n(α={scenarios[scenario_name]["alpha"]}, β={scenarios[scenario_name]["beta"]})', fontweight='bold')
    axes[idx].set_xlabel('Time Period (t)')
    axes[idx].set_ylabel('Number of EVs (R_{ilt})')
    axes[idx].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../output/staggered_evacuation_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

print("Visualization saved to ../output/staggered_evacuation_distributions.png")


## Stochastic Optimization Model with Staggered Evacuation

This section implements the two-stage stochastic optimization model with staggered evacuation constraints:

**Flow Balance Constraints (with R_{ilt})**:

For intermediate charge levels (l ∉ {0, L}):
$$\sum_{(i,j) \in \mathcal{A}} x_{ijlt}^w + y_{jlt}^w + z_{jlt}^w + R_{jlt} = \sum_{(j,k) \in \mathcal{A}} x_{jk,l-1,t+1}^w + y_{j,l+1,t+1}^w + z_{jl,t+1}^w$$

For maximum charge level (l = L):
$$y_{j,L,t}^w + z_{j,L,t}^w + R_{j,L,t} = \sum_{(j,k) \in \mathcal{A}} x_{jk,L-1,t+1}^w + z_{jL,t+1}^w$$

For zero charge level (l = 0):
$$z_{j,0,t}^w + \sum_{(i,j) \in \mathcal{A}} x_{ij,0,t}^w + R_{j,0,t} = z_{j0,t+1}^w + y_{j,1,t+1}^w$$

**Initialization Constraint (z at t=0)**:
$$z_{il0}^w = R_{il0}$$


In [ ]:
# Optimization model function with staggered evacuation (R_ilt)
def primal_run_staggered(
    L, T, cases_num, N, N_s, A, car_num, R_ilt_dict, node_list, road_ca, xi_s, Vi, Vo, B, G, 
    Outputpath, out_folder_name, out_folder, Total_cars, scenario_name=""
):
    """
    Solve two-stage stochastic optimization with staggered evacuation.
    
    Parameters:
    - R_ilt_dict: Dictionary mapping scenario names to R_ilt arrays
    - All other parameters as before
    """
    start = time.time()
    model = pe.ConcreteModel()

    # Model settings
    model.Levels = range(L)
    model.Times = range(4*T+1)
    model.ReleaseTimes = range(4*T)
    model.Cases = range(cases_num)

    model.nodes = pe.Set(initialize=N)
    model.nodes_safe = pe.Set(initialize=N_s)
    model.edges = pe.Set(initialize=A)

    # Parameters: car number initialization
    model.car_num = pe.Param(model.Levels, model.nodes, initialize=car_num)
    
    # Parameters: staggered evacuation R_ilt for current scenario
    # Convert R_ilt array to dictionary format for Pyomo
    R_ilt_param = {}
    R_ilt_current = R_ilt_dict[scenario_name]  # Get R_ilt for current scenario
    for node_idx, node in enumerate(node_list):
        for l in model.Levels:
            for t in model.ReleaseTimes:
                R_ilt_param[(l, node, t)] = float(R_ilt_current[node_idx, l, t])

    model.R_ilt = pe.Param(
        model.Levels,
        model.nodes,
        model.ReleaseTimes,
        initialize=R_ilt_param,
        within=pe.NonNegativeReals
    )

    
    # Road capacity
    model.road_ca = pe.Param(model.edges, model.Cases, model.Times, initialize=road_ca)
    # Charging capacity random damage factor
    model.char_damFac = pe.Param(model.nodes, model.Cases, model.Times, initialize=xi_s)
    # Scenario probabilities
    model.prob = pe.Param(model.Cases, initialize=prob_dict)

    # Define incoming and outgoing sets
    model.Vi = pe.Param(model.nodes, initialize=Vi, default=set(), within=pe.Any)
    model.Vo = pe.Param(model.nodes, initialize=Vo, default=set(), within=pe.Any)

    # VARIABLES
    model.x = pe.Var(model.Levels, model.edges, model.Times, model.Cases, within=pe.NonNegativeReals)
    model.y = pe.Var(model.Levels, model.nodes, model.Times, model.Cases, within=pe.NonNegativeReals)
    model.z = pe.Var(model.Levels, model.nodes, model.Times, model.Cases, within=pe.NonNegativeReals)
    model.s = pe.Var(model.nodes, within=pe.NonNegativeReals)

    # OBJECTIVE VALUE
    obj_func_value = 0

    # CONSTRAINT LISTS
    model.con_flow_bal = pe.ConstraintList()
    model.con_charge_limit = pe.ConstraintList()
    model.con_road_ca = pe.ConstraintList()
    model.budget_con = pe.ConstraintList()
    model.con_x_init = pe.ConstraintList()
    model.con_y_init = pe.ConstraintList()
    model.con_y = pe.ConstraintList()
    model.con_z_init = pe.ConstraintList()

    # 1st stage constraint: BUDGET OF INVESTMENT
    total_budget_expr = sum(G * model.s[j] for j in model.nodes)
    model.budget_con.add(expr=(total_budget_expr <= B))

    # 2nd stage constraints
    for w in range(cases_num):
        # Objective function value
        obj_func_value += model.prob[w] * -sum(
            sum(
                sum(model.z[l, j, t, w] for j in model.nodes_safe) +
                sum(model.y[l, j, t, w] for j in model.nodes_safe)
                for l in model.Levels
            )
            for t in model.Times
        )

        # FLOW BALANCE CONSTRAINTS with R_ilt (staggered evacuation)
        for j in model.nodes:
            for t in range(4*T):
                for l in model.Levels:
                    release_at_t = 0 if t == 0 else model.R_ilt[l, j, t]
                    # Intermediate charge levels (l not in {0, L})
                    if l >= 1 and l <= L-2:
                        flow_in = (
                            sum(model.x[l, i, j, t, w] for i in model.Vi[j]) +
                            model.y[l, j, t, w] +
                            model.z[l, j, t, w] +
                            release_at_t  # STAGGERED START: R_ilt added to inflow
                        )
                        flow_out = (
                            model.y[l+1, j, t+1, w] +
                            model.z[l, j, t+1, w] +
                            sum(model.x[l-1, j, k, t+1, w] for k in model.Vo[j] if k != j)
                        )
                        model.con_flow_bal.add(expr=(flow_in == flow_out))

                    # Minimum charge level (l = 0)
                    if l == 0:
                        flow_in = (
                            sum(model.x[l, i, j, t, w] for i in model.Vi[j]) +
                            model.z[l, j, t, w] +
                            release_at_t  # STAGGERED START: R_ilt added to inflow
                        )
                        flow_out = (
                            model.y[l+1, j, t+1, w] +
                            model.z[l, j, t+1, w]
                        )
                        model.con_flow_bal.add(expr=(flow_in == flow_out))

                    # Maximum charge level (l = L-1)
                    if l == L-1:
                        flow_in = (
                            model.y[l, j, t, w] +
                            model.z[l, j, t, w] +
                            release_at_t  # STAGGERED START: R_ilt added to inflow
                        )
                        flow_out = (
                            sum(model.x[l-1, j, k, t+1, w] for k in model.Vo[j] if k != j) +
                            model.z[l, j, t+1, w]
                        )
                        model.con_flow_bal.add(expr=(flow_in == flow_out))

        # CHARGING LIMIT constraint
        for j in model.nodes:
            for t in model.Times:
                LHS = sum(model.y[l, j, t, w] for l in range(1, L))
                RHS = model.s[j] * model.char_damFac[j, w, t]
                model.con_charge_limit.add(expr=(LHS <= RHS))

        # ROAD CAPACITY constraint
        for (i, j) in model.edges:
            for t in model.Times:
                if i != j:
                    LHS = sum(model.x[l, i, j, t, w] for l in range(L-1))
                    RHS = model.road_ca[i, j, w, t]
                    model.con_road_ca.add(expr=(LHS <= RHS))

        # x INITIALIZATION
        for (i, j) in model.edges:
            for t in model.Times:
                con_expr = (model.x[L-1, i, j, t, w] == 0)
                model.con_x_init.add(expr=con_expr)
            for l in model.Levels:
                con_expr = (model.x[l, i, j, 0, w] == 0)
                model.con_x_init.add(expr=con_expr)

        # y INITIALIZATION
        for j in model.nodes:
            for l in model.Levels:
                con_expr = (model.y[l, j, 0, w] == 0)
                model.con_y_init.add(expr=con_expr)

        # z INITIALIZATION: z_{il0}^w = R_{il0} (STAGGERED START CONSTRAINT)
        for j in model.nodes:
            for l in model.Levels:
                con_expr = (model.z[l, j, 0, w] == model.R_ilt[l, j, 0])
                model.con_z_init.add(expr=con_expr)

        # y constraint for l = 0: cannot charge to level 0
        for j in model.nodes:
            for t in model.Times:
                y_con_expr = (model.y[0, j, t, w] == 0)
                model.con_y.add(expr=y_con_expr)

    model.obj_max_reward = pe.Objective(sense=pe.minimize, expr=obj_func_value)

    # Solve
    result = solver.solve(model, tee=True)

    end = time.time()
    run_time = end - start
    print(f"Execution time: {run_time:.2f} seconds")

    # POST PROCESSING
    s_output = {'Nodes': [], 'CharCap': []}
    for j in model.nodes:
        s_output['Nodes'].append(j)
        s_output['CharCap'].append(model.s[j].value)
    df_s_output = pd.DataFrame(s_output)

    x_output = {'Arcs': [], 'Levels': [], 'Times': [], 'Scenarios': [], 'NumCars': []}
    y_output = {'Nodes': [], 'Levels': [], 'Times': [], 'Scenarios': [], 'NumCars': []}
    z_output = {'Nodes': [], 'Levels': [], 'Times': [], 'Scenarios': [], 'NumCars': []}

    for (i, j) in model.edges:
        for l in model.Levels:
            for t in model.Times:
                for w in model.Cases:
                    x_output['Arcs'].append((i, j))
                    x_output['Levels'].append(l)
                    x_output['Times'].append(t)
                    x_output['Scenarios'].append(w)
                    x_output['NumCars'].append(model.x[l, i, j, t, w].value)

    for j in model.nodes:
        for l in model.Levels:
            for t in model.Times:
                for w in model.Cases:
                    y_output['Nodes'].append(j)
                    z_output['Nodes'].append(j)
                    y_output['Levels'].append(l)
                    z_output['Levels'].append(l)
                    y_output['Times'].append(t)
                    z_output['Times'].append(t)
                    y_output['Scenarios'].append(w)
                    z_output['Scenarios'].append(w)
                    y_output['NumCars'].append(model.y[l, j, t, w].value)
                    z_output['NumCars'].append(model.z[l, j, t, w].value)

    df_x_output = pd.DataFrame(x_output)
    df_y_output = pd.DataFrame(y_output)
    df_z_output = pd.DataFrame(z_output)

    # Save outputs
    if not os.path.exists(out_folder):
        os.makedirs(out_folder)

    df_s_output.to_csv(os.path.join(out_folder, "s_values.csv"), index=False)
    df_x_output.to_csv(os.path.join(out_folder, "x_values.csv"), index=False)
    df_y_output.to_csv(os.path.join(out_folder, "y_values.csv"), index=False)
    df_z_output.to_csv(os.path.join(out_folder, "z_values.csv"), index=False)

    # Calculate evacuation metrics
    num_evac = sum(
        model.prob[w] *
        sum(
            sum((model.y[l, j, 4*T, w].value + model.z[l, j, 4*T, w].value)
                for j in model.nodes_safe)
            for l in model.Levels
        )
        for w in model.Cases
    )

    avg_time = {}
    avg_time_all = {}
    for w in model.Cases:
        suc_cars = sum(
            sum((model.y[l, j, 4*T, w].value + model.z[l, j, 4*T, w].value)
                for j in model.nodes_safe)
            for l in model.Levels
        )
        avg_time[w] = 0
        for t in model.Times:
            if t >= 1:
                avg_time[w] = avg_time[w] + t * (
                    sum(
                        sum((model.y[l, j, t, w].value + model.z[l, j, t, w].value)
                            for j in model.nodes_safe)
                        for l in model.Levels
                    ) -
                    sum(
                        sum((model.y[l, j, t-1, w].value + model.z[l, j, t-1, w].value)
                            for j in model.nodes_safe)
                        for l in model.Levels
                    )
                )
        avg_time_all[w] = (avg_time[w] + (4*T) * (Total_cars - suc_cars)) / Total_cars
        if suc_cars > 0:
            avg_time[w] = avg_time[w] / suc_cars
        else:
            avg_time[w] = 0

    avg_evac_time = sum((model.prob[w] * avg_time[w]) for w in model.Cases)
    avg_evac_time_all = sum((model.prob[w] * avg_time_all[w]) for w in model.Cases)

    return num_evac, avg_evac_time, avg_evac_time_all, run_time


## Run Selected Staggered-Start Scenarios


In [ ]:
# Sensitivity analysis parameters
budget_low = 1
budget_high = 22
budget_list = list(range(budget_low, budget_high, 3))

T_low = 2
T_high = 3
T_list = list(range(T_low, T_high, 1))

Total_cars = 73.203  # Total number of EVs to evacuate

print(f"Budget scenarios: {budget_list}")
print(f"Time window scenarios: {T_list}")


In [ ]:
# Run optimization for all selected staggered-start scenarios
# Edit this list if you only want to run a subset.
scenarios_to_run = ["uniform_evac", "early_evac", "delayed_evac"]

staggered_results = []
staggered_output_root = os.path.join(Outputpath, "staggered_start")
os.makedirs(staggered_output_root, exist_ok=True)

for scenario_to_run in scenarios_to_run:
    print(f"\nRunning optimization for scenario: {scenario_to_run}")
    print("=" * 80)

    out_folder_name = f"staggered_start_{scenario_to_run}_B{B}_T{T}"
    out_folder = os.path.join(staggered_output_root, out_folder_name)

    num_evac, avg_evac_time, avg_evac_time_all, run_time = primal_run_staggered(
        L=L,
        T=T,
        cases_num=cases_num,
        N=N,
        N_s=N_s,
        A=A,
        car_num=car_num,
        R_ilt_dict=R_distributions,
        node_list=node_list,
        road_ca=road_ca,
        xi_s=xi_s,
        Vi=Vi,
        Vo=Vo,
        B=B,
        G=G,
        Outputpath=Outputpath,
        out_folder_name=out_folder_name,
        out_folder=out_folder,
        Total_cars=Total_cars,
        scenario_name=scenario_to_run,
    )

    staggered_results.append({
        "Scenario": scenario_to_run,
        "B": B,
        "EvacTime": T,
        "NumEvac": num_evac,
        "Avg_evac_time": avg_evac_time,
        "Avg_evac_time_all": avg_evac_time_all,
        "RunTime": run_time,
        "OutputFolder": out_folder,
    })

    print(f"\nResults for {scenario_to_run}:")
    print(f"  Number of EVs evacuated: {num_evac:.2f}")
    print(f"  Average evacuation time (successful): {avg_evac_time:.2f}")
    print(f"  Average evacuation time (all): {avg_evac_time_all:.2f}")
    print(f"  Run time: {run_time:.2f} seconds")

staggered_results_df = pd.DataFrame(staggered_results)
staggered_results_csv = os.path.join(staggered_output_root, f"staggered_scenario_results_B{B}_T{T}.csv")
staggered_results_df.to_csv(staggered_results_csv, index=False)

print("\nAll selected staggered scenarios finished.")
print(f"Saved summary results to {staggered_results_csv}")
display(staggered_results_df)


## Plot Staggered Start Sensitivity Results

The cells below generate separate map files for the base case and each staggered-start scenario, plus a cumulative evacuation curve and a compact performance comparison figure.


In [ ]:
# Plotting code for comparing the simultaneous base case with staggered-start scenarios.
# These functions produce:
#   1. Separate capacity allocation map files, one per scenario.
#   2. One cumulative evacuation curve.
#   3. One compact performance comparison figure for NumEvac and Avg_evac_time_all.

from matplotlib.lines import Line2D
from matplotlib.colors import Normalize

COMPARISON_SCENARIO_LABELS = {
    "base_case": "Base case",
    "uniform_evac": "Uniform staggered (alpha=1, beta=1)",
    "early_evac": "Early staggered (alpha=2, beta=5)",
    "delayed_evac": "Delayed staggered (alpha=5, beta=2)",
}

COMPARISON_SCENARIO_ORDER = ["base_case", "early_evac", "uniform_evac", "delayed_evac"]
COMPARISON_COLORS = {
    "base_case": "#222222",
    "early_evac": "#1b9e77",
    "uniform_evac": "#4c78a8",
    "delayed_evac": "#d95f02",
}
COMPARISON_MARKERS = {
    "base_case": "o",
    "early_evac": "s",
    "uniform_evac": "^",
    "delayed_evac": "D",
}


def _clean_scenario_label(scenario_name):
    return COMPARISON_SCENARIO_LABELS.get(str(scenario_name), str(scenario_name).replace("_", " ").title())


def _default_basecase_folder(B_value=B, T_value=T, L_value=L):
    return os.path.join(Outputpath, "even_battery_normal_speed", f"B{B_value}T{T_value}evenL{L_value}")


def _staggered_result_folder(scenario_name, B_value=B, T_value=T):
    folder_name = f"staggered_start_{scenario_name}_B{B_value}_T{T_value}"
    folder = os.path.join(Outputpath, "staggered_start", folder_name)
    if scenario_name == "uniform_evac" and not os.path.exists(folder):
        old_name = f"staggered_start_baseline_B{B_value}_T{T_value}"
        old_folder = os.path.join(Outputpath, "staggered_start", old_name)
        if os.path.exists(old_folder):
            return old_folder
    return folder


def default_comparison_folders(B_value=B, T_value=T, L_value=L):
    return {
        "base_case": _default_basecase_folder(B_value, T_value, L_value),
        "early_evac": _staggered_result_folder("early_evac", B_value, T_value),
        "uniform_evac": _staggered_result_folder("uniform_evac", B_value, T_value),
        "delayed_evac": _staggered_result_folder("delayed_evac", B_value, T_value),
    }


def _load_s_values(result_folder):
    s_path = os.path.join(result_folder, "s_values.csv")
    if not os.path.exists(s_path):
        raise FileNotFoundError(f"Missing {s_path}. Run that scenario first or update result_folders.")
    df_s = pd.read_csv(s_path)
    df_s["Nodes"] = df_s["Nodes"].astype(int)
    df_s["CharCap"] = df_s["CharCap"].fillna(0.0).astype(float)
    return df_s


def _load_flow_outputs(result_folder):
    y_path = os.path.join(result_folder, "y_values.csv")
    z_path = os.path.join(result_folder, "z_values.csv")
    if not os.path.exists(y_path) or not os.path.exists(z_path):
        raise FileNotFoundError(f"Missing y_values.csv or z_values.csv in {result_folder}.")
    return pd.read_csv(y_path), pd.read_csv(z_path)


def _expected_safe_cumulative_by_time(result_folder):
    df_y, df_z = _load_flow_outputs(result_folder)
    safe_nodes = set(int(n) for n in N_s)

    combined = pd.concat([
        df_y[["Nodes", "Times", "Scenarios", "NumCars"]],
        df_z[["Nodes", "Times", "Scenarios", "NumCars"]],
    ], ignore_index=True)
    combined["Nodes"] = combined["Nodes"].astype(int)
    combined = combined[combined["Nodes"].isin(safe_nodes)]

    by_case_time = combined.groupby(["Scenarios", "Times"], as_index=False)["NumCars"].sum()
    by_case_time["Probability"] = by_case_time["Scenarios"].map(prob_dict).fillna(0.0)
    expected = by_case_time.assign(ExpectedCars=by_case_time["NumCars"] * by_case_time["Probability"])
    expected = expected.groupby("Times", as_index=False)["ExpectedCars"].sum()
    expected["Hours"] = expected["Times"] / 4
    return expected.sort_values("Times")


def compute_performance_metrics_from_folder(result_folder, scenario_name, B_value=B, T_value=T, total_cars=Total_cars):
    curve = _expected_safe_cumulative_by_time(result_folder)
    num_evac = float(curve.loc[curve["Times"].idxmax(), "ExpectedCars"])

    increments = curve["ExpectedCars"].diff().fillna(curve["ExpectedCars"])
    times = curve["Times"]
    weighted_time = float((times * increments).sum())
    avg_evac_time = weighted_time / num_evac if num_evac > 0 else 0.0
    avg_evac_time_all = (weighted_time + (4 * T_value) * (total_cars - num_evac)) / total_cars

    return {
        "Scenario": scenario_name,
        "B": B_value,
        "EvacTime": T_value,
        "NumEvac": num_evac,
        "Avg_evac_time": avg_evac_time,
        "Avg_evac_time_all": avg_evac_time_all,
        "OutputFolder": result_folder,
    }


def build_comparison_results(result_folders=None, scenarios_to_plot=None, B_value=B, T_value=T):
    if result_folders is None:
        result_folders = default_comparison_folders(B_value, T_value, L)
    if scenarios_to_plot is None:
        scenarios_to_plot = [s for s in COMPARISON_SCENARIO_ORDER if s in result_folders]

    rows = []
    for scenario_name in scenarios_to_plot:
        rows.append(compute_performance_metrics_from_folder(result_folders[scenario_name], scenario_name, B_value, T_value))
    return pd.DataFrame(rows)


def plot_capacity_allocation_map(
    scenario_name,
    result_folder,
    save_dir="../output/staggered_start_figures",
    vmax=None,
    show_axes=False,
):
    """Create one paper-style charging-capacity allocation map for one scenario."""
    node_xy = df_node[["Nodes", "Long", "Lat"]].copy()
    node_xy["Nodes"] = node_xy["Nodes"].astype(int)
    safe_nodes = set(int(n) for n in N_s)

    df_s = _load_s_values(result_folder)
    plot_df = node_xy.merge(df_s, on="Nodes", how="left")
    plot_df["CharCap"] = plot_df["CharCap"].fillna(0.0)

    if vmax is None:
        vmax = max(float(plot_df["CharCap"].max()), 1e-9)
    norm = Normalize(vmin=0, vmax=vmax)
    cmap = plt.cm.coolwarm

    fig, ax = plt.subplots(figsize=(5.4, 5.6))
    for _, arc in df_arc.iterrows():
        start = node_xy[node_xy["Nodes"] == int(arc["From"])]
        end = node_xy[node_xy["Nodes"] == int(arc["To"])]
        if start.empty or end.empty:
            continue
        ax.plot(
            [start.iloc[0]["Long"], end.iloc[0]["Long"]],
            [start.iloc[0]["Lat"], end.iloc[0]["Lat"]],
            color="0.55",
            linewidth=0.8,
            linestyle="--",
            alpha=0.55,
            zorder=1,
        )

    colors = cmap(norm(plot_df["CharCap"].to_numpy()))
    markers = ["s" if int(n) in safe_nodes else "o" for n in plot_df["Nodes"]]
    for marker in sorted(set(markers)):
        mask = np.array(markers) == marker
        ax.scatter(
            plot_df.loc[mask, "Long"],
            plot_df.loc[mask, "Lat"],
            c=colors[mask],
            s=95 if marker == "o" else 110,
            marker=marker,
            edgecolor="white",
            linewidth=0.75,
            zorder=3,
        )

    for _, row in plot_df.iterrows():
        ax.text(row["Long"], row["Lat"], f"{int(row['Nodes'])}", ha="center", va="center", fontsize=7.5, color="black", zorder=4)
        if row["CharCap"] > 1e-5:
            ax.text(row["Long"], row["Lat"] + 0.012, f"{row['CharCap']:.3f}", ha="center", va="bottom", fontsize=6.5, color="black", zorder=5)

    ax.set_title(_clean_scenario_label(scenario_name), fontsize=11, fontweight="bold")
    ax.set_aspect("equal", adjustable="box")
    if show_axes:
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")
        ax.grid(alpha=0.22, linewidth=0.5)
    else:
        ax.set_axis_off()

    sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Charging capacity installed (thousand cars)")

    legend_handles = [
        Line2D([0], [0], marker="o", color="w", label="Evacuation node", markerfacecolor="0.45", markeredgecolor="white", markersize=8),
        Line2D([0], [0], marker="s", color="w", label="Safe node", markerfacecolor="0.45", markeredgecolor="white", markersize=8),
    ]
    ax.legend(handles=legend_handles, loc="lower right", frameon=True, fontsize=8)

    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f"capacity_allocation_{scenario_name}.png")
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved {scenario_name} capacity allocation map to {save_path}")
    return fig, ax


def plot_capacity_allocation_maps_separately(result_folders=None, scenarios_to_plot=None, save_dir="../output/staggered_start_figures"):
    """Create separate capacity allocation map files with a shared color scale."""
    if result_folders is None:
        result_folders = default_comparison_folders(B, T, L)
    if scenarios_to_plot is None:
        scenarios_to_plot = [s for s in COMPARISON_SCENARIO_ORDER if s in result_folders]

    vmax = 0.0
    for scenario_name in scenarios_to_plot:
        vmax = max(vmax, float(_load_s_values(result_folders[scenario_name])["CharCap"].max()))
    vmax = max(vmax, 1e-9)

    figures = {}
    for scenario_name in scenarios_to_plot:
        figures[scenario_name] = plot_capacity_allocation_map(
            scenario_name,
            result_folders[scenario_name],
            save_dir=save_dir,
            vmax=vmax,
            show_axes=False,
        )
    return figures


def plot_cumulative_evacuation_curve(
    result_folders=None,
    scenarios_to_plot=None,
    save_path="../output/staggered_start_figures/cumulative_evacuation_curve.png",
):
    """Plot expected cumulative EVs evacuated over time for base and staggered cases."""
    if result_folders is None:
        result_folders = default_comparison_folders(B, T, L)
    if scenarios_to_plot is None:
        scenarios_to_plot = [s for s in COMPARISON_SCENARIO_ORDER if s in result_folders]

    fig, ax = plt.subplots(figsize=(7.2, 4.6))
    for scenario_name in scenarios_to_plot:
        curve = _expected_safe_cumulative_by_time(result_folders[scenario_name])
        ax.plot(
            curve["Hours"],
            curve["ExpectedCars"],
            label=_clean_scenario_label(scenario_name),
            color=COMPARISON_COLORS.get(scenario_name),
            marker=COMPARISON_MARKERS.get(scenario_name, "o"),
            linewidth=2.0,
            markersize=5,
        )

    ax.axhline(Total_cars, color="red", linestyle="--", linewidth=1.1, label="Total EVs")
    ax.set_xlabel("Time after evacuation begins (hours)")
    ax.set_ylabel("Expected cumulative EVs evacuated (thousand cars)")
    ax.set_title("Cumulative evacuation comparison", fontsize=12, fontweight="bold")
    ax.grid(alpha=0.3)
    ax.legend(frameon=False, fontsize=8)

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved cumulative evacuation curve to {save_path}")
    return fig, ax


def plot_performance_comparison(
    results_df=None,
    result_folders=None,
    scenarios_to_plot=None,
    save_path="../output/staggered_start_figures/performance_comparison.png",
):
    """Plot compact comparison of NumEvac and Avg_evac_time_all."""
    if results_df is None:
        results_df = build_comparison_results(result_folders, scenarios_to_plot, B, T)
    else:
        results_df = results_df.copy()

    if scenarios_to_plot is None:
        scenarios_to_plot = [s for s in COMPARISON_SCENARIO_ORDER if s in set(results_df["Scenario"])]
    results_df["Scenario"] = pd.Categorical(results_df["Scenario"], categories=scenarios_to_plot, ordered=True)
    results_df = results_df.sort_values("Scenario")

    labels = [_clean_scenario_label(s) for s in results_df["Scenario"].astype(str)]
    colors = [COMPARISON_COLORS.get(s, "0.5") for s in results_df["Scenario"].astype(str)]

    fig, axes = plt.subplots(1, 2, figsize=(9.6, 4.2))
    metrics = [
        ("NumEvac", "Expected EVs evacuated\n(thousand cars)"),
        ("Avg_evac_time_all", "Average evacuation time\n(all EVs, periods)"),
    ]

    base_row = results_df[results_df["Scenario"] == "base_case"]
    for ax, (metric, ylabel) in zip(axes, metrics):
        bars = ax.bar(labels, results_df[metric], color=colors, alpha=0.88, edgecolor="white", linewidth=0.8)
        ax.set_ylabel(ylabel)
        ax.grid(axis="y", alpha=0.3)
        ax.tick_params(axis="x", rotation=25)

        if not base_row.empty:
            base_value = float(base_row.iloc[0][metric])
            for bar, value in zip(bars, results_df[metric]):
                delta = 100 * (value - base_value) / base_value if base_value != 0 else np.nan
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height(),
                    f"{delta:+.1f}%",
                    ha="center",
                    va="bottom",
                    fontsize=8,
                )

    fig.suptitle("Performance comparison: base case versus staggered starts", fontsize=12, fontweight="bold")
    plt.tight_layout()
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved performance comparison figure to {save_path}")
    return fig, axes


# Suggested workflow after the scenario loop has finished:
# comparison_folders = default_comparison_folders(B, T, L)
# plot_capacity_allocation_maps_separately(comparison_folders)
# plot_cumulative_evacuation_curve(comparison_folders)
# comparison_results_df = build_comparison_results(comparison_folders)
# plot_performance_comparison(comparison_results_df)
# display(comparison_results_df)


In [ ]:
comparison_folders = default_comparison_folders(B, T, L)

In [ ]:
plot_capacity_allocation_maps_separately(comparison_folders)

In [ ]:
plot_cumulative_evacuation_curve(comparison_folders)

In [ ]:
comparison_results_df = build_comparison_results(comparison_folders)
plot_performance_comparison(comparison_results_df)
display(comparison_results_df)

## Next Steps

This notebook now provides:
1. ✅ Generation of R_ilt distributions for three Beta scenarios
2. ✅ Statistical analysis of evacuation patterns
3. ✅ Visualization of temporal distributions
4. ✅ Optimization model with staggered evacuation constraints
5. ✅ Scenario loop for uniform_evac, early_evac, and delayed_evac
6. ✅ Comparison plots against the simultaneous base case

To extend the sensitivity analysis, expand `budget_list` and `T_list`, then wrap the scenario loop inside those parameter loops.
